In [22]:
import pandas as pd
import tensorflow as tf
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler, OneHotEncoder, LabelEncoder
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
from tensorflow.keras.callbacks import EarlyStopping
from scikeras.wrappers import KerasClassifier
from sklearn.pipeline import Pipeline
import pickle

In [23]:
df = pd.read_csv('Churn_Modelling.csv')
df.head()

# Preprocess the Data
# Drop irrelevant column
df= df.drop(['RowNumber','CustomerId','Surname'],axis=1)
df.head()

# Encode Categorical Variable
label_encoder_gender = LabelEncoder()
df['Gender'] = label_encoder_gender.fit_transform(df['Gender'])
df.head()

# OneHot Encoding for the "Geography" column
from sklearn.preprocessing import OneHotEncoder
one_hot_encoder_geo = OneHotEncoder()
geo_encode = one_hot_encoder_geo.fit_transform(df[['Geography']])
geo_encode

geo_encoded_df = pd.DataFrame(geo_encode.toarray(),columns=one_hot_encoder_geo.get_feature_names_out(['Geography']))
geo_encoded_df.head()

# Combine the One Hot encoder Column with the original dataset
df = pd.concat([df.drop('Geography',axis=1),geo_encoded_df],axis = 1)
df.head()

# Divide the data into dependent & independent features
X = df.drop('Exited',axis=1)
y = df['Exited']

# Spliting the data into training and testing sets
X_train,X_test,y_train,y_test = train_test_split(X,y,train_size=0.2,random_state=42)

# Scaled the Features
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

# Save the encoder and Scaler
with open('label_encoder_gender.pkl','wb') as file:
    pickle.dump(label_encoder_gender,file)
with open('one_hot_encoder_geo.pkl','wb') as file:
    pickle.dump(one_hot_encoder_geo,file)
# Save the encoder and Scaler
with open('scaler.pkl','wb') as file:
    pickle.dump(scaler,file)

In [24]:
# Define a function to create the model
def create_model(neurons = 32, activation = 'relu', layers = 1):
    model = Sequential()
    model.add(Dense(neurons,activation=activation,input_shape=(X_train.shape[1],)))
    for _ in range(layers-1):
        model.add(Dense(neurons,activation=activation))
    model.add(Dense(1,activation='sigmoid'))
    opt = tf.keras.optimizers.Adam(learning_rate=0.01)
    loss = tf.keras.losses.BinaryCrossentropy()
    model.compile(optimizer=opt,loss=loss,metrics=['accuracy'])
    return model

In [25]:
model = KerasClassifier(layers = 1,neurons=32,activation='relu',
build_fn=create_model,verbose=0,epochs=50,batch_size=10)

In [26]:
# Define grid search parameters
param_grid = {
    'neurons': [16, 32, 64],
    'activation': ['relu', 'tanh'],
    'layers': [1, 2],
    'epochs': [50, 100],
}

In [27]:
grid = GridSearchCV(estimator=model,param_grid=param_grid,cv=3,n_jobs=-1)
grid_result = grid.fit(X_train,y_train)
print(f"Best: {grid_result.best_score_} using {grid_result.best_params_}")

c:\Users\piusd\miniconda3\envs\Churm\Lib\site-packages\scikeras\wrappers.py:925: UserWarning: ``build_fn`` will be renamed to ``model`` in a future release, at which point use of ``build_fn`` will raise an Error instead.
  X, y = self._initialize(X, y)
c:\Users\piusd\miniconda3\envs\Churm\Lib\site-packages\keras\src\layers\core\dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Best: 0.8355026690858774 using {'activation': 'relu', 'epochs': 100, 'layers': 1, 'neurons': 16}
